# 02 — Modeling

Reports Models A and B per spec §8 + §9.2.

**Design note.** Spec §9 states *"none of [the notebooks] refit data or re-call APIs."* The actual fitting lives in `fuel_pred.train.train_models` (run via `make train`), which writes `models/model_{a,b}.pkl`, `models/feature_lists.json`, and the per-fold prediction parquets. This notebook **loads** those artifacts and presents the modeling analysis — it does not refit (re-fitting ~2.3M rows × 2 models would also break the "runs cleanly top-to-bottom" acceptance criterion). The §9.2 section headers ("Fit Model A/B", "Save") are retained for spec alignment, but each is a load-and-report against the trained artifacts.

> Run `make train && make evaluate` first if the pickles / prediction parquets aren't present — every cell below will raise a clear `SystemExit` if an artifact is missing.


## Setup

In [ ]:
from __future__ import annotations

import datetime as dt
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fuel_pred import config


def _require(path: Path) -> Path:
    if not path.exists():
        raise SystemExit(f"{path} not found — run `make train` (+ `make evaluate`) first.")
    return path


features = pd.read_parquet(_require(config.DATA_PROCESSED / "features.parquet"))
features["date"] = pd.to_datetime(features["date"])

with open(_require(config.MODELS_DIR / "model_a.pkl"), "rb") as fh:
    model_a = pickle.load(fh)
with open(_require(config.MODELS_DIR / "model_b.pkl"), "rb") as fh:
    model_b = pickle.load(fh)
with open(_require(config.MODELS_DIR / "feature_lists.json")) as fh:
    feature_lists = json.load(fh)

preds = {
    "test_normal": pd.read_parquet(_require(config.MODELS_DIR / "predictions_test_normal.parquet")),
    "test_crisis": pd.read_parquet(_require(config.MODELS_DIR / "predictions_test_crisis.parquet")),
}
for _df in preds.values():
    _df["date"] = pd.to_datetime(_df["date"])

print(f"features: {len(features):,} rows × {len(features.columns)} cols")
print(f"Model A: {model_a.n_features_in_} features; Model B: {model_b.n_features_in_} features")
for _name, _df in preds.items():
    print(f"{_name}: {len(_df):,} prediction rows")

## 1. Define folds

Time-based, no shuffling. See `fuel_pred.config` for the exact dates.

In [ ]:
# Fold row counts (time-based, no shuffling — spec §8.3). The model trains on
# U91 rows with a non-null t+1 target; the second column reflects that subset.
FOLDS = [
    ("train",       config.SPAN_START,        config.TRAIN_END),
    ("val",         config.VAL_START,         config.VAL_END),
    ("test_normal", config.TEST_START,        config.TEST_NORMAL_END),
    ("test_crisis", config.TEST_CRISIS_START, features["date"].max().date().isoformat()),
]


def fold_label(d: pd.Timestamp) -> str:
    dd = d.date()
    for name, start, end in FOLDS:
        if dt.date.fromisoformat(start) <= dd <= dt.date.fromisoformat(end):
            return name
    return "out_of_span"


features["fold"] = features["date"].map(fold_label)
u91 = features[(features["fuel_code"] == "U91") & features["y_t1"].notna()]
pd.DataFrame({
    "all_rows": features["fold"].value_counts(),
    "U91_with_target": u91["fold"].value_counts(),
}).reindex([f[0] for f in FOLDS]).fillna(0).astype(int)

## 2. Define feature columns

Model A: lag, upstream, calendar, ctx, stn, wx (no `sa2_*`).
Model B: same plus `sa2_*`.

**Identical training rows** for both — only rows where every Model B column is non-null.

In [ ]:
cols_a = feature_lists["A"]["feature_columns"]
cols_b = feature_lists["B"]["feature_columns"]
sa2_in_b = [c for c in cols_b if c.startswith("sa2_")]
print(f"Model A: {len(cols_a)} feature columns (no sa2_*)")
print(f"Model B: {len(cols_b)} feature columns, of which {len(sa2_in_b)} are sa2_*")
print("\nThe SA2 block — the only difference between A and B:")
for c in sa2_in_b:
    print(f"  {c}")

# Identical-rows guard: both models train only where every sa2_* column is
# non-null, so the comparison isolates the SA2 block. Show survival in train.
train_u91 = features[(features["fuel_code"] == "U91")
                     & features["y_t1"].notna()
                     & (features["fold"] == "train")]
mask = train_u91[sa2_in_b].notna().all(axis=1)
print(f"\nidentical-rows guard (train fold): {int(mask.sum()):,} / {len(train_u91):,} "
      f"({100 * mask.mean():.1f}%) rows have all {len(sa2_in_b)} sa2_* non-null")

## 3. Model A (loaded)

Baseline — lag, upstream, calendar, ctx, stn, wx blocks; no `sa2_*`. Hyperparameters per spec §8.2, shared with Model B.


In [ ]:
# Model A summary + the shared hyperparameters (identical for both models).
print("Shared hyperparameters (spec §8.2):")
for k, v in config.LGBM_PARAMS.items():
    print(f"  {k}: {v}")
print()
print(f"Model A: {model_a.n_features_in_} features, "
      f"best_iteration={getattr(model_a, 'best_iteration_', 'n/a')}")

## 4. Model B (loaded)

Identical hyperparameters and training rows as Model A; the only addition is the SA2 block. Top gain-importances side by side below.


In [ ]:
print(f"Model B: {model_b.n_features_in_} features, "
      f"best_iteration={getattr(model_b, 'best_iteration_', 'n/a')}")
print(f"  = Model A's {model_a.n_features_in_} + "
      f"{model_b.n_features_in_ - model_a.n_features_in_} sa2_* features")


def _gain_imp(model, cols: list[str]) -> pd.Series:
    g = model.booster_.feature_importance(importance_type="gain")
    return pd.Series(g, index=cols).sort_values(ascending=False)


imp_a = _gain_imp(model_a, cols_a)
imp_b = _gain_imp(model_b, cols_b)
# reset_index() on an unnamed Series gives columns ["index", 0]; rename explicitly
# (the `names=` kwarg was not consistently supported across pandas versions).
top10 = pd.concat([
    imp_a.head(10).round(0).reset_index().rename(columns={"index": "A_feature", 0: "A_gain"}),
    imp_b.head(10).round(0).reset_index().rename(columns={"index": "B_feature", 0: "B_gain"}),
], axis=1)
top10


## 5. Headline metrics

MAE / RMSE / MAPE / median / p90 absolute error on each test fold.

In [ ]:
from IPython.display import display


def _metrics(y_true: pd.Series, y_pred: pd.Series) -> dict[str, float]:
    err = y_pred - y_true
    ae = err.abs()
    mape = (ae / y_true.abs()).replace([np.inf, -np.inf], np.nan).mean() * 100
    return {
        "MAE": ae.mean(),
        "RMSE": float(np.sqrt((err**2).mean())),
        "MAPE_%": mape,
        "median_AE": float(ae.median()),
        "p90_AE": float(np.percentile(ae, 90)),
    }


rows = []
for fold, df in preds.items():
    for label, col in (("A", "y_pred_a"), ("B", "y_pred_b")):
        rows.append({"fold": fold, "model": label,
                     **{k: round(v, 3) for k, v in _metrics(df["y_true"], df[col]).items()}})
print("Per-model metrics:")
display(pd.DataFrame(rows).set_index(["fold", "model"]))

# Headline Δ (B − A); negative = Model B better.
delta = []
for fold, df in preds.items():
    mae_a = (df["y_pred_a"] - df["y_true"]).abs().mean()
    mae_b = (df["y_pred_b"] - df["y_true"]).abs().mean()
    delta.append({"fold": fold, "MAE_A": round(mae_a, 3), "MAE_B": round(mae_b, 3),
                  "Δ_MAE": round(mae_b - mae_a, 3), "rel_%": round(100 * (mae_b - mae_a) / mae_a, 2)})
print("\nHeadline Δ MAE (negative = augmentor adds value):")
pd.DataFrame(delta).set_index("fold")

## 6. Segmented metrics

By metro/regional, brand, fuel type, SEIFA quintile.

In [ ]:
# Segmented Δ MAE on test_normal. Brand / metro / SEIFA keys aren't in the
# prediction parquets, so join them back from the feature slice.
seg_keys = [c for c in ("station_id", "date", "stn_brand_canonical",
                        "stn_is_metro", "sa2_seifa_irsd_score") if c in features.columns]
seg_slice = features[seg_keys].drop_duplicates(["station_id", "date"])
tn = preds["test_normal"].merge(seg_slice, on=["station_id", "date"], how="left")


def _seg_table(df: pd.DataFrame, group) -> pd.DataFrame:
    d = df.copy()
    d["ae_a"] = (d["y_pred_a"] - d["y_true"]).abs()
    d["ae_b"] = (d["y_pred_b"] - d["y_true"]).abs()
    t = d.groupby(group, observed=True).agg(
        n=("y_true", "size"), MAE_A=("ae_a", "mean"), MAE_B=("ae_b", "mean"))
    t["Δ_MAE"] = t["MAE_B"] - t["MAE_A"]
    return t.round(3)


# SEIFA quintile
try:
    tn["seifa_q"] = pd.qcut(tn["sa2_seifa_irsd_score"], 5,
                            labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
except ValueError:
    tn["seifa_q"] = pd.qcut(tn["sa2_seifa_irsd_score"], 5, duplicates="drop")
print("test_normal — Δ MAE by SEIFA quintile (Q1 = most disadvantaged):")
display(_seg_table(tn, "seifa_q"))

print("test_normal — Δ MAE by brand (top 8 by volume):")
top_brands = tn["stn_brand_canonical"].value_counts().head(8).index
display(_seg_table(tn[tn["stn_brand_canonical"].isin(top_brands)], "stn_brand_canonical")
        .sort_values("Δ_MAE"))

print("Full segmentation (metro/regional, all brands, fuel) → results/comparison.md")

## 7. Residual diagnostics

Residuals over time, check for crisis-period blowup.

In [ ]:
# Residual diagnostics. Residual = prediction − actual.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (a) Model B residual distribution per fold (clipped for readability)
for fold, df in preds.items():
    res = (df["y_pred_b"] - df["y_true"]).clip(-30, 30)
    axes[0].hist(res, bins=60, alpha=0.5, density=True, label=f"{fold} (n={len(res):,})")
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_title("Model B residuals (clipped ±30 c/L)")
axes[0].set_xlabel("pred − actual (c/L)")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# (b) Weekly mean abs error over the whole test span, A vs B — watch for a
# crisis-period blowup.
both = pd.concat(preds.values(), ignore_index=True)
both["week"] = both["date"].dt.to_period("W").dt.start_time
both["ae_a"] = (both["y_pred_a"] - both["y_true"]).abs()
both["ae_b"] = (both["y_pred_b"] - both["y_true"]).abs()
wk = both.groupby("week")[["ae_a", "ae_b"]].mean()
axes[1].plot(wk.index, wk["ae_a"], lw=1.3, label="Model A")
axes[1].plot(wk.index, wk["ae_b"], lw=1.3, label="Model B")
axes[1].axvline(pd.Timestamp(config.TEST_CRISIS_START), color="red", ls="--",
                alpha=0.6, label="crisis start")
axes[1].set_title("Weekly mean abs error across the test span")
axes[1].set_ylabel("MAE (c/L)")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Both models' error rises sharply at the 2026 crisis boundary (expected — "
      "OOD regime). Model B tracks at or below Model A throughout.")

## 8. Artifact provenance

Where the loaded artifacts come from (the notebook reads, it does not write — spec §9).


In [ ]:
# Artifacts are produced by the pipeline, not this notebook:
#
#   make train     -> models/model_a.pkl, models/model_b.pkl,
#                     models/feature_lists.json,
#                     models/predictions_test_{normal,crisis}.parquet
#   make evaluate  -> results/comparison.md  (full segmentation + importance
#                     + SA2<->non-SA2 correlation tables)
#
# This notebook loaded those and reproduced the headline + segment views.
# See results/README.md for the synthesised narrative and notebook 03 /
# results/shap/ for explainability.
print("Loaded artifacts:")
for p in ("model_a.pkl", "model_b.pkl", "feature_lists.json",
          "predictions_test_normal.parquet", "predictions_test_crisis.parquet"):
    fp = config.MODELS_DIR / p
    print(f"  [{'ok' if fp.exists() else 'MISSING'}] {fp}")
comp = config.RESULTS_DIR / "comparison.md"
print(f"  [{'ok' if comp.exists() else 'MISSING'}] {comp}")